In [46]:
# magic commands to reload custom modules
%load_ext autoreload
%autoreload 2

# magic for matplotlib
%matplotlib notebook

import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import uncertainties.unumpy as unp
import uncertainties as unc
from uncertainties import ufloat, ufloat_fromstr

from scipy.optimize import curve_fit
from mapp_tricks.peakfit import parse_spectrum_file, plot_plotly, PeakFitter


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [47]:
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import curve_fit

data = np.loadtxt(
    "./flat-samples_mcmc_progress_2026-09-14_17-17-33_SRIM.csv",
    delimiter=","
)

hist_data = data[:, 0]


def gaussian(x, amplitude, mean, stddev):
    return amplitude * np.exp(-0.5 * ((x - mean) / stddev) ** 2)


# Histogram as probability density
counts, edges = np.histogram(hist_data, bins=200, density=True)
centers = 0.5 * (edges[:-1] + edges[1:])


# Fit only around 10 MeV
fit_mask = (centers >= 8.0) & (centers <= 11.0)

x_fit = centers[fit_mask]
y_fit = counts[fit_mask]

# Initial guess: amplitude, mean, standard deviation
p0 = [
    y_fit.max(),
    10.0,
    0.2,
]

popt, pcov = curve_fit(
    gaussian,
    x_fit,
    y_fit,
    p0=p0,
    bounds=(
        [0     , 9.0 , 0],
        [np.inf, 11.0, 0.9],
    ),
)

amplitude, mean, stddev = popt
errors = np.sqrt(np.diag(pcov))

print(f"Mean     = {mean:.5f} ± {errors[1]:.5f} MeV")
print(f"Std. dev = {stddev:.5f} ± {errors[2]:.5f} MeV")


# Smooth Gaussian for plotting
x_gauss = np.linspace(8, 15, 500)
y_gauss = gaussian(x_gauss, *popt)


fig = go.Figure()

fig.add_trace(go.Histogram(
    x=hist_data,
    nbinsx=200,
    histnorm="probability density",
    name="Samples",
    opacity=0.7,
))

fig.add_trace(go.Scatter(
    x=x_gauss,
    y=y_gauss,
    mode="lines",
    name=(
        f"Gaussian fit: "
        f"μ = {mean:.3f} MeV, "
        f"σ = {stddev:.3f} MeV"
    ),
    line=dict(width=3),
))

fig.update_layout(
    xaxis_title="Energy [MeV]",
    yaxis_title="Density [1/MeV]",
)

fig.show()

Mean     = 10.01028 ± 0.01995 MeV
Std. dev = 0.70486 ± 0.02381 MeV
